In [1]:
import pandas as pd
import numpy as np
from csi_utils import csi_databases
import os
from concurrent.futures import ThreadPoolExecutor
import glob
from pathlib import Path

In [2]:
def generate_friendly_list(items):
    if len(items) > 1:
        sql_list = [(item,) for item in items]
    else:
        sql_list = [(items[0],)]
    return sql_list


In [3]:
df = pd.read_csv('../data/raw/locked_slides.csv')
slides  = df["slide_id"].unique()
slides = generate_friendly_list(slides)

In [4]:
db = csi_databases.DatabaseHandler("/mnt/secrets/database/reader.yaml", is_production=True)
query = (
        "select slide_id, frame_id, x, y, type, manual_classification "
        "from ocular_hitlist where slide_id = %s"
    )
query_2 = ("select reportv2_link from analysis where slide_id = %s")
hitlist_data = pd.DataFrame()
for slide in slides:
    try:
        slides_df = db.get(query, slide)
        path = db.get(query_2, slide)
        df = pd.DataFrame(
                [list(row) for row in slides_df['results']],
                columns=slides_df['headers']
            )
        df["path"] = path["results"][0][0].replace('report-v2.html', '')
        hitlist_data = pd.concat([hitlist_data, df])
    except Exception as e:
        print(f"Error processing slide {slide}: {e}")
hitlist_data = hitlist_data.rename(columns = {
    "slide_id": "slide_id",
    "frame_id": "tile",
    "x": "x",
    "y": "y",
    "type": "biotype",
    "manual_classification": "classification"
    })
hitlist_data = hitlist_data.dropna(subset=["classification"])

In [5]:
hitlist_data.to_csv("./data/hitlist_data.csv", index=False)


In [2]:
hitlist_data = pd.read_csv("./data/hitlist_data.csv")

In [3]:
# Remove the substring and add the prefix
hitlist_data['path'] = '/mnt' + hitlist_data['path']

In [4]:
df_for_path_check = hitlist_data[['slide_id', 'path']].copy()
# Drop duplicates in slide_id
df_for_path_check.drop_duplicates(subset='slide_id', inplace=True)

In [5]:
def check_files_in_directory(directory):
    if os.path.isdir(directory):
        tif_file = Path( directory+"Tile000001.tif")
        jpg_file = Path(directory+"Tile000001.jpg")
        if tif_file.exists():
            return True, False
        elif jpg_file.exists():
            return False, True
        else:
            return False, False

In [6]:
# Function to apply the check in parallel
def apply_parallel(df, func):
    with ThreadPoolExecutor() as executor:
        results = list(executor.map(func, df['path']))
    df['tif'], df['jpg'] = zip(*results)
    return df

In [7]:
# Apply the function to the DataFrame in parallel
df_for_path_check = apply_parallel(df_for_path_check, check_files_in_directory)

In [8]:
# Merge the DataFrames on 'slide_id'
hitlist_data = hitlist_data.merge(df_for_path_check[['slide_id', 'tif', 'jpg']], on='slide_id', how='left')

# Add the 'ext' column based on conditions
def determine_ext(row):
    if row['tif']:
        return 'tif'
    elif row['jpg']:
        return 'jpg'
    else:
        return None

hitlist_data['ext'] = hitlist_data.apply(determine_ext, axis=1)
hitlist_data = hitlist_data.drop(columns=['tif', 'jpg'])

In [9]:
hitlist_data["classification"].unique()

array(['(Dapi-)CK|V|CD', '(Dapi-)CK', '(Dapi-)CK|CD', 'D|CK|V|CD',
       '(Dapi-)CD', '(Dapi-)CK|V', 'D|CK|CD', 'D|CK|V', 'D|V', '(Dapi-)V',
       'D|V|CD', 'D', 'D|CD', 'D|CK', 'Dapi- Cell', '(Dapi-)V|CD',
       '(DAPI-)CD', nan, '(DAPI-)V|CD', '(DAPI-)CK|V|CD', '(DAPI-)CK|CD',
       'DAPI+|FITC+|CY5+', 'DAPI+', 'DAPI+|FITC+|TRITC+|CY5+', '(DAPI-)V',
       '(DAPI-)CK', 'DAPI+|CY5+', 'DAPI+|FITC+', '(DAPI-)CK|V',
       'ONCOSOME', 'DAPI+|TRITC+|CY5+'], dtype=object)

In [10]:
# Remove oncosomes
hitlist_data = hitlist_data[hitlist_data['classification'] != 'ONCOSOME']
# Remove Dapi- Cell
hitlist_data = hitlist_data[hitlist_data['classification'] != 'Dapi- Cell']

In [11]:
# Replace 'Dapi+' or 'DAPI+' with 'D'
hitlist_data['classification'] = hitlist_data['classification'].str.replace('Dapi+', 'D', regex=False)
hitlist_data['classification'] = hitlist_data['classification'].str.replace('DAPI+', 'D', regex=False)
# Replace '(Dapi-)' or '(DAPI-)' with 'D|'
hitlist_data['classification'] = hitlist_data['classification'].str.replace('(Dapi-)', '', regex=False)
hitlist_data['classification'] = hitlist_data['classification'].str.replace('(DAPI-)', '', regex=False)
# Remove nan values
hitlist_data = hitlist_data.dropna(subset=['classification'])

In [12]:
hitlist_data["classification"].unique()

array(['CK|V|CD', 'CK', 'CK|CD', 'D|CK|V|CD', 'CD', 'CK|V', 'D|CK|CD',
       'D|CK|V', 'D|V', 'V', 'D|V|CD', 'D', 'D|CD', 'D|CK', 'V|CD',
       'D|FITC+|CY5+', 'D|FITC+|TRITC+|CY5+', 'D|CY5+', 'D|FITC+',
       'D|TRITC+|CY5+'], dtype=object)

In [13]:
# Replace '' and 'NA' with NaN
hitlist_data.replace({'': np.nan, 'NA': np.nan}, inplace=True)
# Drop rows with NaN values
hitlist_data.dropna(inplace=True)

In [14]:
hitlist_data["classification"].unique()

array(['CK|V|CD', 'CK', 'CK|CD', 'D|CK|V|CD', 'CD', 'CK|V', 'D|CK|CD',
       'D|CK|V', 'D|V', 'V', 'D|V|CD', 'D', 'D|CD', 'D|CK', 'V|CD',
       'D|FITC+|CY5+', 'D|FITC+|TRITC+|CY5+', 'D|CY5+', 'D|FITC+',
       'D|TRITC+|CY5+'], dtype=object)

In [15]:
# Replace substrings
hitlist_data['classification'] = hitlist_data['classification'].str.replace('FITC+', 'V', regex=False)
hitlist_data['classification'] = hitlist_data['classification'].str.replace('CY5+', 'CD', regex=False)
hitlist_data['classification'] = hitlist_data['classification'].str.replace('TRITC+', 'CK', regex=False)

In [16]:
hitlist_data["classification"].unique()

array(['CK|V|CD', 'CK', 'CK|CD', 'D|CK|V|CD', 'CD', 'CK|V', 'D|CK|CD',
       'D|CK|V', 'D|V', 'V', 'D|V|CD', 'D', 'D|CD', 'D|CK', 'V|CD',
       'D|V|CK|CD'], dtype=object)

In [20]:
class_map = {
    'D': 'D',
    'CK': 'CK',
    'CD': 'CD',
    'V': 'V',
    'CK|V|CD': 'CK|CD|V',
    'CK|CD': 'CK|CD',
    'D|CK|V|CD': 'D|CK|CD|V',
    'CK|V': 'CK|V',
    'D|CK|CD': 'D|CK|CD',
    'D|CK|V': 'D|CK|V',
    'D|V': 'D|V',
    'D|V|CD': 'D|CD|V',
    'D|CD': 'D|CD',
    'D|CK': 'D|CK',
    'V|CD': 'CD|V',
    'D|V|CK|CD': 'D|CK|CD|V'
}

In [21]:
hitlist_data['classification'] = hitlist_data['classification'].replace(class_map)

In [22]:
hitlist_data["classification"].unique()

array(['CK|CD|V', 'CK', 'CK|CD', 'D|CK|CD|V', 'CD', 'CK|V', 'D|CK|CD',
       'D|CK|V', 'D|V', 'V', 'D|CD|V', 'D', 'D|CD', 'D|CK', 'CD|V'],
      dtype=object)

In [34]:
for cls in hitlist_data["classification"].unique():
    os.makedirs(f"./data/interim/{cls}", exist_ok=True)

In [24]:
from skimage.io import imread, imsave
from PIL import Image
from skimage import exposure, img_as_ubyte

import numpy as np
from tqdm import tqdm

In [25]:
def crop(image, x, y, imagesize):
    """
    Crop a grayscale image to the specified size with (x, y) at the center.
    Apply zero padding if the crop goes out of bounds.

    Parameters:
    - image: 2D numpy array representing the grayscale image.
    - x: The x-coordinate of the center of the crop.
    - y: The y-coordinate of the center of the crop.
    - imagesize: The size of the crop (width and height).

    Returns:
    - Cropped image as a 2D numpy array with zero padding if necessary.
    """
    half_size = imagesize // 2

    # Calculate the crop boundaries
    x_start = x - half_size
    x_end = x + half_size
    y_start = y - half_size
    y_end = y + half_size

    # Create an empty image with zeros (zero padding)
    cropped_image = np.zeros((imagesize, imagesize), dtype=image.dtype)

    # Calculate the valid crop boundaries within the original image
    valid_x_start = max(x_start, 0)
    valid_x_end = min(x_end, image.shape[1])
    valid_y_start = max(y_start, 0)
    valid_y_end = min(y_end, image.shape[0])

    # Calculate the corresponding positions in the cropped image
    crop_x_start = max(0, -x_start)
    crop_x_end = crop_x_start + (valid_x_end - valid_x_start)
    crop_y_start = max(0, -y_start)
    crop_y_end = crop_y_start + (valid_y_end - valid_y_start)

    # Copy the valid part of the original image to the cropped image
    cropped_image[crop_y_start:crop_y_end, crop_x_start:crop_x_end] = image[valid_y_start:valid_y_end, valid_x_start:valid_x_end]

    return img_as_ubyte(cropped_image)

In [26]:
def get_event(path, tile, x, y, ext):
    try:
        dapi_path = path+"Tile"+str(tile).rjust(6, "0")+f".{ext}"
        ck_path = path+"Tile"+str(tile+2304).rjust(6,"0")+f".{ext}"
        cd45_path = path+"Tile"+str(tile+2304*2).rjust(6,"0")+f".{ext}"
        fitc_path = path+"Tile"+str(tile+2304*4).rjust(6,"0")+f".{ext}"
        
        if not os.path.exists(dapi_path):
            raise FileNotFoundError(f"File not found: {dapi_path}")
        if not os.path.exists(ck_path):
            raise FileNotFoundError(f"File not found: {ck_path}")
        if not os.path.exists(cd45_path):
            raise FileNotFoundError(f"File not found: {cd45_path}")
        if not os.path.exists(fitc_path):
            raise FileNotFoundError(f"File not found: {fitc_path}")
        
        dapi = imread(dapi_path)
        ck = imread(ck_path)
        cd45 = imread(cd45_path)
        fitc = imread(fitc_path)

        event = np.zeros((50,50,4), dtype=np.uint8)
        for i, img in enumerate([dapi, ck, cd45, fitc]):
            event[:,:,i] = crop(img, x, y, 50)
        return event

    except FileNotFoundError as e:
        print(e)
        return None
    except Exception as e:
        print(f"Error reading images: {e}")
        return None
    


In [27]:
def get_composite(dapi, ck, cd45, fitc):
    dtype = dapi.dtype
    max_val = np.iinfo(dapi.dtype).max
    dapi = dapi.astype(np.float32)
    ck = ck.astype(np.float32)
    cd45 = cd45.astype(np.float32)
    fitc = fitc.astype(np.float32)
    rgb = np.zeros((dapi.shape[0], dapi.shape[1], 3),
                   dtype='float')
    rgb[...,0] = ck+fitc
    rgb[...,1] = cd45+fitc
    rgb[...,2] = dapi.astype(np.float32)+fitc
    rgb[rgb > max_val] = max_val
    rgb = rgb.astype(dtype)
    return rgb

In [35]:
def process_row(row, imgsize=50):
    if type(row) == tuple:
        row = row[1]
    try:
        slide_id = row["slide_id"]
        path = row["path"]
        tile = row["tile"]
        x = row["x"]
        y = row["y"]
        ext = row["ext"]
        crop = get_event(path, tile, x, y, ext)
        label = row["classification"]

        if crop is not None:
            composite = get_composite(crop[:,:,0], crop[:,:,1], crop[:,:,2], crop[:,:,3])
        if composite is not None:
            imsave(f"./data/interim/{label}/{slide_id}_{tile}_{x}_{y}.png", composite)
    except Exception as e:
        print(e)
    

In [36]:
def process_events(hitlist, imgsize = 50):
    print(f"Generating images for {len(hitlist)} events")
    no_of_events = len(hitlist)
    hitlist.apply(lambda row: process_row(row, imgsize=50), axis=1)

In [ ]:
results = process_events(hitlist_data, imgsize=50)

Generating images for 612382 events


/tmp/ipykernel_209602/789998373.py:17: UserWarning: ./data/interim/D/0AFE102_1894_458_764.png is a low contrast image
  imsave(f"./data/interim/{label}/{slide_id}_{tile}_{x}_{y}.png", composite)
/tmp/ipykernel_209602/789998373.py:17: UserWarning: ./data/interim/D/0AFE201_237_1273_907.png is a low contrast image
  imsave(f"./data/interim/{label}/{slide_id}_{tile}_{x}_{y}.png", composite)
/tmp/ipykernel_209602/789998373.py:17: UserWarning: ./data/interim/D/0AFE201_507_1328_390.png is a low contrast image
  imsave(f"./data/interim/{label}/{slide_id}_{tile}_{x}_{y}.png", composite)


File not found: /mnt/csidata/OncoScope/tubeID_0AFE2/exptID_16250/slideID_0AFE201/bzScanner/proc/Tile010400.jpg
cannot access local variable 'composite' where it is not associated with a value
File not found: /mnt/csidata/OncoScope/tubeID_0AFE2/exptID_16250/slideID_0AFE201/bzScanner/proc/Tile010411.jpg
cannot access local variable 'composite' where it is not associated with a value
File not found: /mnt/csidata/OncoScope/tubeID_0AFE2/exptID_16250/slideID_0AFE201/bzScanner/proc/Tile010414.jpg
cannot access local variable 'composite' where it is not associated with a value
File not found: /mnt/csidata/OncoScope/tubeID_0AFE2/exptID_16250/slideID_0AFE201/bzScanner/proc/Tile010424.jpg
cannot access local variable 'composite' where it is not associated with a value
File not found: /mnt/csidata/OncoScope/tubeID_0AFE2/exptID_16250/slideID_0AFE201/bzScanner/proc/Tile010438.jpg
cannot access local variable 'composite' where it is not associated with a value
File not found: /mnt/csidata/OncoScope/t

/tmp/ipykernel_209602/789998373.py:17: UserWarning: ./data/interim/D/0AFD302_1923_666_32.png is a low contrast image
  imsave(f"./data/interim/{label}/{slide_id}_{tile}_{x}_{y}.png", composite)
/tmp/ipykernel_209602/789998373.py:17: UserWarning: ./data/interim/D/0AFD302_2103_994_925.png is a low contrast image
  imsave(f"./data/interim/{label}/{slide_id}_{tile}_{x}_{y}.png", composite)
